In [2]:
# src/sh_ssw_methods/methods/ozone_threshold.py
from __future__ import annotations
import xarray as xr
import pandas as pd
from utils import (
    cos_weighted_mean,
    remove_doy_climatology,
    remove_yearly_mean, 
    build_events_df, 
    group_consecutive_dates,
    enforce_min_gap,
    apply_persistence
)



In [27]:
import numpy as np

In [45]:
def detect_ozone_threshold(
    tco3: xr.DataArray,
    *,
    lat_band=(-90.0,-65.0),
    thresh_du=40.0,
    min_persist_days=3,
    min_gap_days=20,
    time_dim="time",
    lat_dim="lat", 
    data_source: str | None = None
):

    # raise error if input is not xr.DataArray
    if not isinstance(tco3, xr.DataArray):
        raise TypeError(
            "detect_ozone_threshold expects an xarray.DataArray. "
            "If you have a Dataset, select a variable first, e.g. ds['tcO3']."
        )

    # polar cap weighted
    cap = cos_weighted_mean(tco3.sel({lat_dim: slice(*lat_band)}), dim=lat_dim)

    # compute anom from daily climatology
    anom = remove_doy_climatology(cap, time_dim)

    # remove trend by removing yearly mean
    fin = remove_yearly_mean(anom)
    
    # filter out dates that satisfy threshold
    ind = fin.where(fin > thresh_du, drop=True)

    # group consecutive days
    counts = group_consecutive_dates(ind['time'].to_index())

    # test persistence criterion, need to last at least for 3 days
    persist = apply_persistence(counts, min_persist_days = 3)

    # enforce minimum gap, events between need to separate at least 20 days
    tpersist = enforce_min_gap(persist, min_gap_days=20)

    event_dates = pd.to_datetime(tpersist["first"])

    # also look up corresponding ozone polar cap anomalies
    series_for_value = fin          
    
    # robust alignment: convert to pandas Series and reindex on event dates
    s_val = pd.Series(
        series_for_value.values,
        index=pd.to_datetime(series_for_value[time_dim].to_index())
    )
    ozone_onset = s_val.reindex(event_dates).to_numpy()  # value at event 'first' day
    
    
    # output into df
    # Ozone threshold events (dates and ozone value at onset)
    events_df = build_events_df(
        dates=event_dates,
        method="ozone_threshold", #### change here the name later
        definition=f"ozone_du{int(thresh_du)}_{int(lat_band[0])}to{int(lat_band[1])}S",
        data_source=data_source or "",
        threshold=f"{thresh_du}DU",
        lat_band=str(list(lat_band)),
        notes=f"min_persist={min_persist_days}d; min_gap={min_gap_days}d",
        extra_cols={
        "ozone_onset_DU": ozone_onset, }, 
    )
    
    return events_df, event_dates.values

1. ozone threshold

In [6]:
import os

In [7]:
years = ("1980", "2020")
lat_band=(-90.0,-65.0)
thresh_du=40.0
min_persist_days=3
min_gap_days=20
data_source="MERRA2"

In [9]:
# open sample total column ozone data
tco3_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"tco3_MERRA2.nc"
xtco3 = xr.open_dataset(tco3_path)

In [10]:
xtco3

<xarray.Dataset> Size: 5MB
Dimensions:  (time: 16802, lat: 73)
Coordinates:
  * time     (time) datetime64[ns] 134kB 1979-07-01 1979-07-02 ... 2025-06-30
  * lat      (lat) float32 292B -90.0 -87.5 -85.0 -82.5 ... 82.5 85.0 87.5 90.0
Data variables:
    tcO3     (time, lat) float32 5MB ...

In [46]:
da = xtco3['tcO3']

In [47]:
events_df, event_dates = detect_ozone_threshold(da)

In [48]:
events_df

,date,method,definition,data_source,threshold,level_hpa,latitude,lat_band,notes,ozone_onset_DU
0,1980-10-07,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,40.631332
1,1980-11-25,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,41.745819
2,1981-09-21,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,43.621754
3,1981-11-19,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,41.341801
4,1982-10-04,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,44.327801
5,1983-11-03,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,43.989090
6,1984-11-01,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,42.813858
7,1986-09-05,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,42.850536
8,1986-11-13,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,48.368191
9,1988-09-24,ozone_threshold,ozone_du40_-90to-65S,,40.0DU,None,None,"[-90.0, -65.0]",min_persist=3d; min_gap=20d,42.315155


In [35]:
lat_dim = 'lat'
time_dim = 'time'

In [22]:
# polar cap weighted
cap = cos_weighted_mean(da.sel({lat_dim: slice(*lat_band)}), dim=lat_dim)

# compute anom from daily climatology
anom = remove_doy_climatology(cap, time_dim)

# remove trend by removing yearly mean
fin = remove_yearly_mean(anom)

# filter out dates that satisfy threshold
ind = fin.where(fin > thresh_du, drop=True)

# group consecutive days
counts = group_consecutive_dates(ind['time'].to_index())

# test persistence criterion, need to last at least for 3 days
persist = apply_persistence(counts, min_persist_days = 3)

# enforce minimum gap, events between need to separate at least 20 days
tpersist = enforce_min_gap(persist, min_gap_days=20)

event_dates = pd.to_datetime(tpersist["first"])

In [28]:
# pick which series to record (processed anomalies, or raw)
series_for_value = fin           # or `cap` if you want raw polar-cap DU

# robust alignment: convert to pandas Series and reindex on event dates
s_val = pd.Series(
    series_for_value.values,
    index=pd.to_datetime(series_for_value[time_dim].to_index())
)
ozone_onset = s_val.reindex(event_dates).to_numpy()  # value at event 'first' day



In [30]:
ozone_onset

array([40.631332, 41.74582 , 43.621754, 41.3418  , 44.3278  , 43.98909 ,
       42.813858, 42.850536, 48.36819 , 42.315155, 44.80738 , 41.137863,
       41.053337, 43.285107, 60.686817, 42.866863, 41.054165, 41.7055  ,
       41.446358, 44.235622, 41.646736, 41.34445 , 41.989555],
      dtype=float32)

In [18]:
fin

<xarray.DataArray 'tcO3' (time: 16802)> Size: 67kB
array([nan, nan, nan, ..., nan, nan, nan], dtype=float32)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1979-07-01 1979-07-02 ... 2025-06-30
    dayofyear  (time) int64 134kB 182 183 184 185 186 ... 177 178 179 180 181
    year       (time) int64 134kB 1979 1979 1979 1979 ... 2025 2025 2025 2025

In [19]:
fin.where(fin > thresh_du, drop=True)

<xarray.DataArray 'tcO3' (time: 364)> Size: 1kB
array([ 40.631332,  44.72061 ,  45.27388 ,  40.017715,  40.29927 ,
        41.74582 ,  47.041077,  51.788788,  50.90146 ,  40.986603,
        43.621754,  51.343174,  50.370564,  50.92377 ,  47.96523 ,
        46.81769 ,  41.718037,  41.68871 ,  41.171055,  41.3418  ,
        51.166233,  54.31931 ,  56.45856 ,  49.24146 ,  45.86289 ,
        43.294132,  44.3278  ,  52.666485,  55.993298,  55.728405,
        50.248837,  53.938183,  55.43481 ,  58.799343,  52.931118,
        49.33801 ,  44.6692  ,  45.121914,  42.12544 ,  45.4835  ,
        45.795177,  43.158443,  45.765472,  44.277786,  43.98909 ,
        43.06627 ,  45.961105,  50.022903,  56.391647,  55.377792,
        57.095932,  51.311234,  44.626022,  41.771896,  42.813858,
        43.55748 ,  45.362167,  44.057632,  46.127823,  53.11769 ,
        46.616776,  41.451218,  42.850536,  46.80058 ,  47.858288,
        40.400448,  40.14004 ,  48.36819 ,  65.93728 ,  77.79064 ,
        74.6426  ,  69.760765,  67.1625  ,  64.147514,  65.41311 ,
        54.43539 ,  45.273342,  42.495144,  40.217434,  40.39354 ,
        43.141327,  42.315155,  54.791336,  62.689484,  64.4368  ,
        59.629135,  56.581482,  54.64438 ,  51.564133,  49.823822,
        50.704025,  47.778305,  41.373306,  41.373184,  41.506622,
        47.91472 ,  51.695297,  59.627625,  63.961212,  72.4402  ,
...
        59.784203,  58.489525,  53.62638 ,  47.467674,  48.01171 ,
        47.802666,  43.721382,  42.849693,  47.08541 ,  47.453514,
        46.52053 ,  41.81707 ,  44.235622,  53.632565,  57.898617,
        53.242702,  46.59408 ,  40.24002 ,  41.646736,  43.92002 ,
        46.639442,  47.737404,  46.420753,  42.89109 ,  41.34445 ,
        41.918137,  42.14882 ,  44.792496,  46.671555,  48.87422 ,
        48.35164 ,  43.835587,  41.989555,  41.52619 ,  41.834877,
        45.232872,  48.44744 ,  52.19526 ,  56.10769 ,  62.098137,
        66.957664,  78.07601 ,  87.148384,  88.527016,  88.74493 ,
        84.704704,  73.50135 ,  60.641228,  50.90473 ,  46.940666,
        42.68186 ,  41.155296,  46.116295,  54.10755 ,  64.24941 ,
        76.770195,  80.25895 ,  68.846855,  67.21842 ,  65.86439 ,
        60.22001 ,  59.54212 ,  61.629173,  59.108376,  53.79812 ,
        45.264214,  41.166267,  46.413673,  48.117516,  47.34025 ,
        45.015358,  43.08851 ,  40.979713,  49.237206,  59.165855,
        63.566444,  62.49494 ,  58.390617,  56.81437 ,  56.00962 ,
        55.331276,  52.00135 ,  48.75712 ,  45.796883,  45.713966,
        45.612007,  51.438972,  58.095253,  60.98075 ,  60.319954,
        56.595222,  52.837105,  51.167366,  48.899178,  46.84571 ,
        47.64292 ,  47.64457 ,  45.264442,  41.119545], dtype=float32)
Coordinates:
  * time       (time) datetime64[ns] 3kB 1980-10-07 1980-10-08 ... 2019-11-22
    dayofyear  (time) int64 3kB 281 282 283 284 285 330 ... 322 323 324 325 326
    year       (time) int64 3kB 1980 1980 1980 1980 1980 ... 2019 2019 2019 2019

In [20]:
counts = group_consecutive_dates(ind['time'].to_index())

In [21]:
counts

,first,last,count
0,1980-10-07,1980-10-11,5
1,1980-11-25,1980-11-29,5
2,1981-09-21,1981-09-29,9
3,1981-11-19,1981-11-25,7
4,1982-10-04,1982-10-15,12
5,1982-10-21,1982-10-24,4
6,1983-10-03,1983-10-04,2
7,1983-11-03,1983-11-12,10
8,1984-11-01,1984-11-08,8
9,1986-09-05,1986-09-07,3
